# Clase 2 — Machine Learning, algoritmos y evaluación

## Pregunta central

> **¿Cómo sabemos si un modelo aprendió algo útil y no solo memorizó?**

## Idea principal

Un modelo útil supera un baseline en datos no vistos y se evalúa con métricas alineadas al costo del error.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Reconocer muestras, features y labels.
- Distinguir clasificación, regresión, clustering y refuerzo.
- Comparar KNN, árbol y regresión logística.
- Separar train, validation y test evitando leakage.
- Interpretar métricas de clasificación y regresión.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Datos y tipos de aprendizaje |
| 2 | Entrenamiento y generalización |
| 3 | Clasificación comparada |
| 4 | Métricas y costo del error |
| 5 | Regresión y baseline |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. No es necesario implementar algoritmos desde cero.
5. Si aparece un término nuevo, buscá primero su definición en el glosario de la clase.

**Conexión con el programa:** ML predictivo del Track Salud y evaluación de modelos en ambos tracks.

## Glosario mínimo

| Término | Explicación breve |
|---|---|
| Muestra | Un caso individual del dataset |
| Feature | Información usada como entrada |
| Label | Respuesta conocida que queremos predecir |
| Algoritmo | Procedimiento que ajusta o usa un modelo |
| Modelo | Función con parámetros aprendidos |
| Loss | Error numérico que el entrenamiento intenta reducir |
| Baseline | Referencia simple que debemos superar |
| Train | Datos usados para ajustar parámetros |
| Validation | Datos usados para comparar decisiones |
| Test | Evaluación final sobre casos reservados |
| Métrica | Número que resume un aspecto del desempeño |
| Generalización | Capacidad de funcionar sobre casos nuevos |
| Leakage | Uso accidental de información que no estaría disponible |

---
## 1. Anatomía de un dataset

Un dataset es una colección estructurada de casos. En datos
tabulares, cada fila suele representar una muestra y cada columna
una variable.

```text
             features de entrada                 label
             -------------------                 -----
muestra 1 -> edad, consultas, cobertura          riesgo
muestra 2 -> edad, consultas, cobertura          riesgo
muestra 3 -> edad, consultas, cobertura          riesgo
```

| Concepto | Pregunta que responde | Ejemplo |
|---|---|---|
| Muestra | ¿Cuál es el caso individual? | Un paciente, una parcela o una imagen |
| Feature | ¿Qué sabe el modelo sobre el caso? | Edad, banda espectral, palabra o píxel |
| Label | ¿Cuál era la respuesta esperada? | Riesgo, especie o clase correcta |
| `X` | ¿Dónde agrupamos las entradas? | Matriz de muestras por features |
| `y` | ¿Dónde agrupamos las respuestas? | Vector de labels |

Las features pueden ser:

- **numéricas:** edad, temperatura o cantidad;
- **categóricas:** región, tipo de dispositivo o especialidad;
- **binarias:** sí/no, presente/ausente;
- **secuenciales:** tokens o muestras de audio;
- **espaciales:** píxeles o bandas de un ráster.

El modelo no conoce el significado humano de una columna. Solo
recibe una representación numérica. Por eso la documentación y el
preprocesamiento son parte del contrato.

### Tipos de aprendizaje

| Tipo | Qué información recibe | Qué intenta aprender | Ejemplo |
|---|---|---|---|
| Supervisado | Entradas y respuestas conocidas | Relación entre `X` e `y` | Clasificación o regresión |
| No supervisado | Entradas sin labels | Estructura o grupos | Clustering |
| Por refuerzo | Estado, acciones y recompensas | Política de acciones | Control de un sistema |

En este notebook trabajamos con supervisado porque permite observar
claramente predicción, error y evaluación. “Supervisado” no
significa que una persona mire cada inferencia: significa que
durante el entrenamiento existen respuestas de referencia.

### Qué es feature engineering

Feature engineering es crear o transformar entradas para expresar
información útil de una forma que el algoritmo pueda aprovechar:

| Dato disponible | Feature posible |
|---|---|
| Fecha y hora | Día de semana, turno o estacionalidad |
| Historial de eventos | Cantidad en los últimos 30 días |
| Dos bandas espectrales | Un índice calculado entre ambas |

La transformación debe poder repetirse en inferencia y no usar
información futura. En el Track Salud se profundizará este trabajo
sobre datos clínicos; aquí solo reconocemos su lugar en el pipeline.

```text
dato crudo -> transformación documentada -> feature
                                        |
                                        +-> misma operación en producción
```

## 2. Entrenamiento, validación y test

```
dataset completo
├── train       -> ajusta parámetros
├── validation  -> elige algoritmo e hiperparámetros
└── test        -> estima final sobre casos reservados
```

La separación existe porque evaluar con los mismos casos usados
para aprender produce una impresión demasiado optimista.

| Conjunto | El modelo puede aprender de él | Para qué se usa |
|---|---:|---|
| Train | Sí | Ajustar parámetros |
| Validation | No | Comparar alternativas durante desarrollo |
| Test | No | Realizar una evaluación final |

### Generalización, underfitting y overfitting

```text
underfitting
train: bajo       validation: bajo
el modelo no captura el patrón

ajuste razonable
train: alto       validation: parecido
el patrón se sostiene fuera de train

overfitting
train: muy alto   validation: claramente menor
el modelo memorizó detalles de train
```

La diferencia entre el resultado de train y validation se suele
llamar **generalization gap**. No existe un valor universalmente
correcto: se interpreta junto con el nivel de desempeño y el
problema.

### Data leakage

Leakage ocurre cuando el entrenamiento recibe información que no
debería tener. Ejemplos:

- normalizar usando el promedio de todo el dataset antes del split;
- incluir una variable creada después del evento que queremos predecir;
- tener muestras de la misma persona en train y test;
- elegir repetidamente el modelo mirando test.

Varias transformaciones también aprenden información de los datos:

| Componente | Qué aprende de train |
|---|---|
| Scaler o escalador | Promedio, dispersión o rango para cambiar la escala |
| Imputador | Valor con el que reemplazará datos faltantes |
| Selector de features | Qué variables conservará |

Esos componentes se ajustan solo con train. Un `Pipeline` es un
objeto que encadena transformaciones y modelo en un orden fijo; al
entrenarlo ayuda a aplicar la misma secuencia sin calcular
estadísticas sobre validation o test.

La **loss** guía el ajuste de parámetros durante entrenamiento. Una
**métrica** resume el comportamiento que nos interesa evaluar. Pueden
coincidir, pero no son necesariamente lo mismo.

```text
loss       -> señal usada para ajustar el modelo
métrica    -> número usado para interpretar el resultado
costo real -> consecuencia del error en el proceso
```

---
## 3. Experimento de clasificación

Clasificar significa elegir una categoría entre alternativas
conocidas. En un problema binario usaremos las clases `0` y `1`.
El algoritmo aprende una **frontera de decisión**: una separación
entre regiones del espacio de features.

```text
feature 2
    ^
    |   0  0       región predicha como clase 0
    |  0  0
    |         frontera
    |            1  1
    |          1  1       región predicha como clase 1
    +----------------------------> feature 1
```

Usaremos dos features visibles y tres algoritmos principales.

### K-Nearest Neighbors

KNN no construye una fórmula global. Guarda ejemplos y, para un
caso nuevo:

1. calcula qué ejemplos están más cerca;
2. selecciona los `k` vecinos más próximos;
3. decide por la clase más frecuente entre ellos.

Si una feature usa valores de 0 a 1 y otra de 0 a 100 000, la
segunda dominará las distancias. Por eso KNN suele necesitar
escalado.

### Árbol de decisión

Un árbol crea preguntas sucesivas:

```text
¿feature 1 < umbral?
      /          \
    sí            no
    |             |
otra pregunta   clase 1
```

Es fácil de inspeccionar, pero un árbol sin límites puede crear
muchas ramas y memorizar train.

### Regresión logística

Aunque se llama “regresión”, se usa para clasificación. Calcula un
score a partir de una combinación de features y lo transforma en
una probabilidad entre 0 y 1. Un **threshold** o umbral,
habitualmente 0.5, convierte esa probabilidad en una clase: por
encima predice la clase positiva y por debajo la negativa.

| Algoritmo | Intuición | Hiperparámetro visible | Riesgo inicial |
|---|---|---|---|
| KNN | Votan los vecinos | Cantidad `k` | Sensible a escala y ruido |
| Árbol | Encadena preguntas | Profundidad | Memorizar train |
| Regresión logística | Separación lineal probabilística | Regularización | No capturar fronteras curvas |

La **regularización** agrega una preferencia por parámetros menos
extremos para reducir el riesgo de memorizar train. No garantiza
generalización, pero controla parte de la complejidad del modelo.

El dataset de lunas es sintético y tiene dos dimensiones para que
podamos ver las fronteras. No representa un problema real.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    root_mean_squared_error,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

FAST_MODE = True
SEED = 42

X, y = make_moons(n_samples=420, noise=0.26, random_state=SEED)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25, stratify=y_train_val, random_state=SEED
)

print("Train:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))

In [ ]:
modelos = {
    "Baseline mayoritaria": DummyClassifier(strategy="most_frequent"),
    "KNN": Pipeline([
        ("escala", StandardScaler()),
        ("modelo", KNeighborsClassifier(n_neighbors=9)),
    ]),
    "Árbol": DecisionTreeClassifier(max_depth=4, random_state=SEED),
    "Árbol sin límite": DecisionTreeClassifier(random_state=SEED),
    "Regresión logística": Pipeline([
        ("escala", StandardScaler()),
        ("modelo", LogisticRegression(random_state=SEED)),
    ]),
}

filas = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    filas.append({
        "modelo": nombre,
        "accuracy_train": accuracy_score(
            y_train, modelo.predict(X_train)
        ),
        "accuracy_validation": accuracy_score(
            y_val, modelo.predict(X_val)
        ),
    })

comparacion = pd.DataFrame(filas)
comparacion["gap_train_validation"] = (
    comparacion["accuracy_train"]
    - comparacion["accuracy_validation"]
)
comparacion = comparacion.sort_values(
    "accuracy_validation", ascending=False
)
comparacion.round(3)

In [ ]:
def dibujar_frontera(ax, modelo, titulo):
    margen = 0.55
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - margen, X[:, 0].max() + margen, 220),
        np.linspace(X[:, 1].min() - margen, X[:, 1].max() + margen, 220),
    )
    grilla = np.c_[xx.ravel(), yy.ravel()]
    pred = modelo.predict(grilla).reshape(xx.shape)
    ax.contourf(xx, yy, pred, alpha=0.23, cmap="coolwarm")
    ax.scatter(
        X_val[:, 0], X_val[:, 1], c=y_val,
        cmap="coolwarm", edgecolor="white", s=35
    )
    ax.set_title(titulo)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, nombre in zip(
    axes, ["KNN", "Árbol", "Regresión logística"]
):
    dibujar_frontera(ax, modelos[nombre], nombre)
plt.tight_layout()
plt.show()

### Leer el resultado

La tabla muestra tres valores:

| Columna | Interpretación |
|---|---|
| `accuracy_train` | Aciertos sobre casos usados para aprender |
| `accuracy_validation` | Aciertos sobre casos no usados para ajustar |
| `gap_train_validation` | Diferencia entre ambos resultados |

Para interpretar:

1. compará cada modelo con el baseline;
2. observá validation antes que train;
3. buscá una diferencia grande entre train y validation;
4. relacioná la forma de la frontera con el algoritmo.

Una frontera flexible puede capturar patrones curvos, pero también
puede seguir puntos aislados y ruido. El árbol sin límite permite
observar ese riesgo.

Validation sirve para elegir durante el desarrollo. Test no se mira
repetidamente porque, si lo usamos para muchas decisiones, deja de
funcionar como examen independiente.

## 4. Métricas de clasificación

Primero debemos decidir qué clase llamaremos **positiva**. “Positiva”
no significa buena: es la clase de interés que queremos detectar.

La matriz de confusión organiza cuatro resultados:

| Real | Predicción | Nombre | Interpretación |
|---|---|---|---|
| Positivo | Positivo | Verdadero positivo | Detectamos el caso |
| Negativo | Negativo | Verdadero negativo | Descartamos correctamente |
| Negativo | Positivo | Falso positivo | Generamos una falsa alarma |
| Positivo | Negativo | Falso negativo | Omitimos un caso real |

```text
                  PREDICCIÓN
              negativo   positivo
REAL negativo     TN         FP
     positivo     FN         TP
```

A partir de esos conteos:

- **Accuracy:** proporción total de aciertos.
- **Precision:** de las predicciones positivas, cuántas eran correctas.
- **Recall:** de los positivos reales, cuántos encontró el modelo.
- **F1:** resumen que combina precision y recall.

Accuracy puede engañar en clases desbalanceadas. Si solo 1 de cada
100 casos es positivo, predecir siempre “negativo” logra 99 % de
accuracy y 0 % de recall para la clase importante.

| Si queremos reducir... | Suele importar más... |
|---|---|
| Casos reales omitidos | Recall |
| Falsas alarmas | Precision |
| Ambos tipos y necesitamos un resumen | F1 |
| Todos los errores pesan parecido | Accuracy |

En salud, un falso negativo y un falso positivo rara vez tienen el
mismo costo. En visión ocurre lo mismo con objetos omitidos o falsas
detecciones. La métrica es una decisión del problema, no una
propiedad universal del algoritmo.

In [ ]:
mejor_nombre = comparacion.iloc[0]["modelo"]
mejor_modelo = modelos[mejor_nombre]
pred_test = mejor_modelo.predict(X_test)

metricas_cls = pd.Series({
    "accuracy": accuracy_score(y_test, pred_test),
    "precision": precision_score(y_test, pred_test),
    "recall": recall_score(y_test, pred_test),
    "f1": f1_score(y_test, pred_test),
})
matriz = confusion_matrix(y_test, pred_test)

print("Modelo elegido usando validation:", mejor_nombre)
display(metricas_cls.round(3).to_frame("test"))
display(pd.DataFrame(
    matriz,
    index=["real 0", "real 1"],
    columns=["pred 0", "pred 1"],
))

---
## 5. Regresión: cuando la salida es un número

En clasificación elegimos categorías. En regresión estimamos un
valor numérico continuo:

```text
features -> modelo de regresión -> número estimado

horas y día -> demanda esperada -> 37.4 consultas
bandas      -> cobertura        -> 62.1 %
```

La regresión lineal busca una recta —o un plano cuando hay varias
features— que aproxime la relación entre entrada y salida.

```text
valor real
    ^
    |             punto
    |         punto
    |     -------- recta estimada
    |  punto
    +------------------------> feature
```

El **residuo** es la diferencia entre el valor real y el estimado.
Como casi nunca coincide exactamente, no preguntamos “¿acertó?”, sino
“¿qué tan lejos quedó?”.

### Métricas de regresión

| Métrica | Qué resume | Cómo se interpreta |
|---|---|---|
| MAE | Distancia absoluta promedio | Mismas unidades que la salida |
| RMSE | Distancia que enfatiza errores grandes | Mismas unidades, más sensible a extremos |
| R² | Comparación con predecir la media | 1 es mejor; 0 equivale al baseline de media |

Un R² negativo es posible: significa que el modelo fue peor que
predecir siempre la media sobre ese conjunto.

En la práctica comparamos una regresión lineal con un
`DummyRegressor`. El baseline no intenta aprender la relación:
predice siempre el promedio de train.

In [ ]:
rng = np.random.default_rng(SEED)
horas = rng.uniform(0, 10, 180).reshape(-1, 1)
demanda = 12 + 4.2 * horas[:, 0] + rng.normal(0, 4.0, len(horas))

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    horas, demanda, test_size=0.25, random_state=SEED
)
regresores = {
    "Baseline media": DummyRegressor(strategy="mean"),
    "Regresión lineal": LinearRegression(),
}
filas_reg = []
for nombre, modelo in regresores.items():
    modelo.fit(X_reg_train, y_reg_train)
    pred = modelo.predict(X_reg_test)
    filas_reg.append({
        "modelo": nombre,
        "MAE": mean_absolute_error(y_reg_test, pred),
        "RMSE": root_mean_squared_error(y_reg_test, pred),
        "R2": r2_score(y_reg_test, pred),
    })

pd.DataFrame(filas_reg).round(3)

## Actividad — elegir la métrica por el costo

Una métrica no se elige porque sea popular. Se elige según la salida
y el costo del error.

Para cada fila:

1. identificá si la salida es una clase, un número o una región;
2. describí el error que más daño produce;
3. elegí una métrica que haga visible ese error;
4. explicá qué información adicional necesitarías antes de aceptar
   el modelo.

Cambiá una respuesta o agregá un caso. La justificación debe mencionar
qué error queremos evitar.

La fila de segmentación anticipa **IoU** (*Intersection over Union*):
mide qué proporción de superposición existe entre una región predicha
y la región correcta. La calcularemos visualmente en la clase 5.

In [ ]:
decisiones = pd.DataFrame([
    ["Detectar pacientes de alto riesgo", "recall",
     "Queremos reducir casos positivos omitidos."],
    ["Evitar alarmas innecesarias", "precision",
     "Queremos que las alertas positivas sean confiables."],
    ["Predecir cantidad de consultas", "MAE",
     "Necesitamos error en las mismas unidades."],
    ["Comparar segmentaciones", "IoU",
     "Necesitamos medir superposición de máscaras."],
], columns=["problema", "metrica_inicial", "justificacion"])

# TODO: modificá una decisión y explicá el costo del error.
decisiones

---

## Síntesis de la clase

- Los algoritmos representan fronteras y relaciones diferentes.
- Un baseline evita celebrar un modelo que no aporta valor.
- Train aprende, validation compara y test examina.
- La métrica se elige según el costo del error.
- Clasificación y regresión comparten pipeline, pero no métricas.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

La clase 3 muestra cómo texto, imagen, audio y datos geoespaciales se convierten en representaciones numéricas.

## Conexión con los tracks

Track Salud profundizará clasificación/regresión clínica; Track Imagen reutilizará splits, generalización y métricas sobre píxeles y objetos.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.